# Week 6 — Transformer Error Analysis

This notebook is **post-training inference only**. It does not retrain any model.

It loads the five saved DistilBERT checkpoints, recreates the corresponding MELD test examples, generates standardized predictions, and saves:

- `predictions.csv`
- `metrics.json`
- `confusion_matrix.png`
- `classification_report.csv`

The five experiments are:

1. transformer baseline
2. context-1
3. context-3
4. context-5
5. context-1 + speaker

**Non-negotiable MELD rule:** context/adjacency must never cross a dialogue boundary or a missing `Utterance_ID` gap.

Run this notebook locally first to validate the project/data pipeline. The actual trained checkpoints can then be loaded from Google Drive in Colab.


In [1]:
from pathlib import Path
import json
import inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
)

from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("Python / PyTorch / Transformers ready")
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Python / PyTorch / Transformers ready
Torch: 2.13.0+cu130
CUDA available: False


In [2]:
# Colab-only setup
# This cell is harmless locally: it simply does nothing outside Colab.

IN_COLAB = False

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    drive.mount("/content/drive")
    DRIVE_CHECKPOINT_ROOT = Path("/content/drive/MyDrive/checkpoints")
    print("Colab detected.")
    print("Drive checkpoints:", DRIVE_CHECKPOINT_ROOT)
else:
    print("Local environment detected.")


Local environment detected.


In [3]:
# Project paths
PROJECT_ROOT = Path.cwd()

# If the notebook is opened from notebooks/, move to the repository root.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "meld"
RESULTS_DIR = PROJECT_ROOT / "outputs" / "experiments"
LOCAL_CHECKPOINT_DIR = PROJECT_ROOT / "outputs" / "models"

print("Project root:", PROJECT_ROOT)
print("MELD data:", DATA_DIR)
print("Results:", RESULTS_DIR)
print("Local checkpoints:", LOCAL_CHECKPOINT_DIR)

assert (DATA_DIR / "test_sent_emo.csv").exists(), (
    "MELD test CSV not found. Run this notebook from the project root "
    "or from the notebooks/ directory."
)


Project root: /home/chitta/Projects/emotion-dynamics-nlp
MELD data: /home/chitta/Projects/emotion-dynamics-nlp/data/raw/meld
Results: /home/chitta/Projects/emotion-dynamics-nlp/outputs/experiments
Local checkpoints: /home/chitta/Projects/emotion-dynamics-nlp/outputs/models


In [4]:
LABELS = [
    "anger",
    "disgust",
    "fear",
    "joy",
    "neutral",
    "sadness",
    "surprise",
]

LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}

print(LABEL2ID)


{'anger': 0, 'disgust': 1, 'fear': 2, 'joy': 3, 'neutral': 4, 'sadness': 5, 'surprise': 6}


## 1. Load MELD test data

We use the official test split only. No training or validation data is used here.

For the baseline, every test utterance is eligible.

For context experiments, the target subset must match the context construction used during Week 5.


In [5]:
test_df = pd.read_csv(DATA_DIR / "test_sent_emo.csv")

required_columns = [
    "Utterance",
    "Speaker",
    "Emotion",
    "Dialogue_ID",
    "Utterance_ID",
]

missing = [c for c in required_columns if c not in test_df.columns]
assert not missing, f"Missing MELD columns: {missing}"

test_df = test_df.rename(columns={
    "Utterance": "text",
    "Speaker": "speaker",
    "Emotion": "emotion",
    "Dialogue_ID": "dialogue_id",
    "Utterance_ID": "utterance_id",
})

test_df = test_df.sort_values(
    ["dialogue_id", "utterance_id"]
).reset_index(drop=True)

assert test_df["text"].notna().all()
assert test_df["emotion"].isin(LABELS).all()

print("Test utterances:", len(test_df))
print("Dialogues:", test_df["dialogue_id"].nunique())
test_df.head()


Test utterances: 2610
Dialogues: 280


,Sr No.,text,speaker,emotion,Sentiment,dialogue_id,utterance_id,Season,Episode,StartTime,EndTime
0,1,Why do all you’re coffee mugs have numbers on ...,Mark,surprise,positive,0,0,3,19,"00:14:38,127","00:14:40,378"
1,2,Oh. That’s so Monica can keep track. That way ...,Rachel,anger,negative,0,1,3,19,"00:14:40,629","00:14:47,385"
2,3,Y'know what?,Rachel,neutral,neutral,0,2,3,19,"00:14:56,353","00:14:57,520"
3,19,"Come on, Lydia, you can do it.",Joey,neutral,neutral,1,0,1,23,"0:10:44,769","0:10:46,146"
4,20,Push!,Joey,joy,positive,1,1,1,23,"0:10:46,146","0:10:46,833"


## 2. Recreate context examples

The project already has `src.data.context.build_context_examples`.

We use that existing implementation rather than writing a second context-construction algorithm.

If the local function signature differs, the next cell prints it so we can adjust the adapter rather than silently changing the experiment definition.


In [6]:
from src.data.context import build_context_examples

print(inspect.signature(build_context_examples))


(df: pandas.DataFrame, context_turns: int = 1, include_speaker: bool = False) -> pandas.DataFrame


In [7]:
def make_context_examples(df, context_turns, include_speaker=False):
    """Compatibility wrapper around the project's existing context builder."""
    kwargs = {}

    signature = inspect.signature(build_context_examples)
    params = signature.parameters

    # The Week 5 implementation is expected to accept these names.
    candidates = {
        "context_turns": context_turns,
        "include_speaker": include_speaker,
        "speaker_tokens": include_speaker,
    }

    for name, value in candidates.items():
        if name in params:
            kwargs[name] = value

    # Prefer the existing implementation and expose failures clearly.
    result = build_context_examples(df, **kwargs)

    if isinstance(result, pd.DataFrame):
        out = result.copy()
    else:
        out = pd.DataFrame(result)

    print("Context builder returned:", out.shape)
    print("Columns:", list(out.columns))
    return out


### Context sanity check

Run this before loading any model.

We need to verify that the generated context data actually contains the target label and text used by the Week 5 training pipeline.


In [8]:
sample_context = make_context_examples(
    test_df,
    context_turns=1,
    include_speaker=False,
)

display(sample_context.head())


Context builder returned: (2328, 6)
Columns: ['dialogue_id', 'utterance_id', 'speaker', 'text', 'emotion', 'context']


,dialogue_id,utterance_id,speaker,text,emotion,context
0,0,1,Rachel,Oh. That’s so Monica can keep track. That way ...,anger,Why do all you’re coffee mugs have numbers on ...
1,0,2,Rachel,Y'know what?,neutral,Oh. That’s so Monica can keep track. That way ...
2,1,1,Joey,Push!,joy,"Come on, Lydia, you can do it.\nPush!"
3,1,2,Joey,"Push 'em out, push 'em out, harder, harder.",joy,"Push!\nPush 'em out, push 'em out, harder, har..."
4,1,3,Joey,"Push 'em out, push 'em out, way out!",joy,"Push 'em out, push 'em out, harder, harder.\nP..."


In [9]:
# Inspect likely text/label columns so we do not silently guess.
print("Columns:", list(sample_context.columns))


Columns: ['dialogue_id', 'utterance_id', 'speaker', 'text', 'emotion', 'context']


In [10]:
# Validate context examples before model inference

assert len(sample_context) == 2328, (
    f"Expected 2328 Context-1 test examples, got {len(sample_context)}"
)

required_columns = {
    "dialogue_id",
    "utterance_id",
    "speaker",
    "text",
    "emotion",
    "context",
}

assert required_columns.issubset(sample_context.columns), (
    f"Missing columns: {required_columns - set(sample_context.columns)}"
)

assert sample_context["text"].notna().all()
assert sample_context["emotion"].notna().all()
assert sample_context["context"].notna().all()

print("Context validation passed.")
print(f"Examples: {len(sample_context)}")
print(f"Emotion classes: {sample_context['emotion'].nunique()}")
print(sorted(sample_context["emotion"].unique()))

Context validation passed.
Examples: 2328
Emotion classes: 7
['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']


In [11]:
print("Sample context:")
print(sample_context.iloc[0]["context"])

print("\nTarget utterance:")
print(sample_context.iloc[0]["text"])

print("\nTarget emotion:")
print(sample_context.iloc[0]["emotion"])

Sample context:
Why do all you’re coffee mugs have numbers on the bottom?
Oh. That’s so Monica can keep track. That way if one on them is missing, she can be like, ‘Where’s number 27?!’

Target utterance:
Oh. That’s so Monica can keep track. That way if one on them is missing, she can be like, ‘Where’s number 27?!’

Target emotion:
anger


## 3. Normalize context examples

This adapter accepts the project's existing context DataFrame and normalizes common field names.

If an expected field is missing, this cell intentionally stops rather than guessing.


In [ ]:
def normalize_examples(df):
    df = df.copy()

    text_candidates = ["text", "input_text", "context_text", "model_input"]
    label_candidates = ["emotion", "label", "target", "target_label"]

    text_col = next((c for c in text_candidates if c in df.columns), None)
    label_col = next((c for c in label_candidates if c in df.columns), None)

    if text_col is None or label_col is None:
        raise ValueError(
            "Could not identify text/label columns. "
            f"Available columns: {list(df.columns)}"
        )

    out = df.copy()
    out["model_text"] = out[text_col].astype(str)

    if pd.api.types.is_numeric_dtype(out[label_col]):
        out["label_id"] = out[label_col].astype(int)
        out["true_emotion"] = out["label_id"].map(ID2LABEL)
    else:
        out["true_emotion"] = out[label_col].astype(str)
        out["label_id"] = out["true_emotion"].map(LABEL2ID)

    if out["label_id"].isna().any():
        bad = out.loc[out["label_id"].isna(), "true_emotion"].unique()
        raise ValueError(f"Unknown emotion labels: {bad}")

    return out.reset_index(drop=True)

baseline_examples = normalize_examples(test_df)
print(baseline_examples[["model_text", "true_emotion", "label_id"]].head())


## 4. Checkpoint configuration

### Local validation

If a trained checkpoint has already been copied into `outputs/models/`, point `LOCAL_CHECKPOINT_DIR` at it.

### Colab

For the actual inference run, set `DRIVE_CHECKPOINT_ROOT` to the mounted Google Drive `checkpoints/` directory.

The Drive checkpoint structure shown during Week 5 is:

```text
checkpoints/
├── transformer_baseline/
├── transformer_context_1/
├── transformer_context_3/
├── transformer_context_5/
└── transformer_context_1_speaker/
```

Each `best/` directory contains `config.json`, `model.safetensors`, tokenizer files, and training configuration.


In [ ]:
# Change this in Colab after mounting Drive.
DRIVE_CHECKPOINT_ROOT = None

EXPERIMENTS = {
    "transformer_baseline": {
        "context_turns": 0,
        "include_speaker": False,
    },
    "transformer_context_1": {
        "context_turns": 1,
        "include_speaker": False,
    },
    "transformer_context_3": {
        "context_turns": 3,
        "include_speaker": False,
    },
    "transformer_context_5": {
        "context_turns": 5,
        "include_speaker": False,
    },
    "transformer_context_1_speaker": {
        "context_turns": 1,
        "include_speaker": True,
    },
}

def find_checkpoint(experiment_name):
    # Prefer an explicitly mounted Google Drive checkpoint.
    if DRIVE_CHECKPOINT_ROOT is not None:
        root = Path(DRIVE_CHECKPOINT_ROOT) / experiment_name / "best"
        if (root / "config.json").exists() and (root / "model.safetensors").exists():
            return root

    # Then look for a local checkpoint.
    local_candidates = [
        LOCAL_CHECKPOINT_DIR / experiment_name / "best",
        LOCAL_CHECKPOINT_DIR / experiment_name,
    ]

    for root in local_candidates:
        if (root / "config.json").exists() and (root / "model.safetensors").exists():
            return root

    return None


In [ ]:
# Verify which checkpoints are available.
for name in EXPERIMENTS:
    print(f"{name}: {find_checkpoint(name)}")


## 5. Inference utilities

Inference preserves the model's probability distribution. We need probabilities later for trajectory and turning-point analysis, not only hard labels.


In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_checkpoint(checkpoint_path):
    checkpoint_path = Path(checkpoint_path)

    tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint_path)

    model.to(DEVICE)
    model.eval()

    return tokenizer, model


@torch.no_grad()
def predict_examples(examples, checkpoint_path, batch_size=16, max_length=128):
    tokenizer, model = load_checkpoint(checkpoint_path)

    texts = examples["model_text"].tolist()
    all_probs = []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )

        encoded = {k: v.to(DEVICE) for k, v in encoded.items()}

        logits = model(**encoded).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()

        all_probs.append(probs)

    probabilities = np.vstack(all_probs)
    predictions = probabilities.argmax(axis=1)

    return predictions, probabilities


In [ ]:
def evaluate_predictions(examples, predictions):
    y_true = examples["label_id"].to_numpy()
    y_pred = np.asarray(predictions)

    accuracy = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    weighted_f1 = f1_score(y_true, y_pred, average="weighted")

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=list(range(len(LABELS))),
        zero_division=0,
    )

    report = pd.DataFrame({
        "emotion": LABELS,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "support": support,
    })

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(len(LABELS))),
    )

    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "classification_report": report,
        "confusion_matrix": cm,
    }


def save_experiment_outputs(name, examples, predictions, probabilities, metrics):
    out_dir = RESULTS_DIR / name
    out_dir.mkdir(parents=True, exist_ok=True)

    pred_df = examples.copy()
    pred_df["predicted_id"] = predictions
    pred_df["predicted_emotion"] = [ID2LABEL[i] for i in predictions]
    pred_df["correct"] = pred_df["label_id"].to_numpy() == predictions

    for i, label in ID2LABEL.items():
        pred_df[f"prob_{label}"] = probabilities[:, i]

    pred_df.to_csv(out_dir / "predictions.csv", index=False)

    metrics_json = {
        "experiment": name,
        "test_examples": int(len(examples)),
        "test_accuracy": float(metrics["accuracy"]),
        "test_macro_f1": float(metrics["macro_f1"]),
        "test_weighted_f1": float(metrics["weighted_f1"]),
    }

    with open(out_dir / "metrics.json", "w") as f:
        json.dump(metrics_json, f, indent=2)

    metrics["classification_report"].to_csv(
        out_dir / "classification_report.csv",
        index=False,
    )

    fig, ax = plt.subplots(figsize=(8, 8))
    im = ax.imshow(metrics["confusion_matrix"])
    ax.set_xticks(range(len(LABELS)))
    ax.set_yticks(range(len(LABELS)))
    ax.set_xticklabels(LABELS, rotation=45, ha="right")
    ax.set_yticklabels(LABELS)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"Confusion Matrix — {name}")

    for i in range(len(LABELS)):
        for j in range(len(LABELS)):
            ax.text(j, i, str(metrics["confusion_matrix"][i, j]),
                    ha="center", va="center")

    fig.colorbar(im)
    plt.tight_layout()
    fig.savefig(out_dir / "confusion_matrix.png", dpi=300)
    plt.show()

    return pred_df


## 6. Run one experiment first

Do **not** immediately run all five.

Start with the baseline. Once the baseline loads and produces predictions correctly, run the four contextual experiments.


In [ ]:
# Baseline first
baseline_checkpoint = find_checkpoint("transformer_baseline")

if baseline_checkpoint is None:
    print(
        "No baseline checkpoint found locally. "
        "This is expected until the checkpoint is available in outputs/models/ "
        "or Google Drive is mounted in Colab."
    )
else:
    predictions, probabilities = predict_examples(
        baseline_examples,
        baseline_checkpoint,
        batch_size=16,
        max_length=128,
    )

    metrics = evaluate_predictions(baseline_examples, predictions)

    print(f"Accuracy:    {metrics['accuracy']:.4f}")
    print(f"Macro F1:    {metrics['macro_f1']:.4f}")
    print(f"Weighted F1: {metrics['weighted_f1']:.4f}")

    baseline_predictions = save_experiment_outputs(
        "transformer_baseline",
        baseline_examples,
        predictions,
        probabilities,
        metrics,
    )

    baseline_predictions.head()


## 7. Run all available experiments

For context models, the examples are recreated using the project's existing context builder.

The notebook intentionally reports the number of test examples for each experiment because context windows naturally have smaller eligible test sets.


In [ ]:
all_results = []
prediction_frames = {}

for name, config in EXPERIMENTS.items():
    checkpoint = find_checkpoint(name)

    if checkpoint is None:
        print(f"SKIP {name}: checkpoint not available")
        continue

    if config["context_turns"] == 0:
        examples = baseline_examples
    else:
        raw_examples = make_context_examples(
            test_df,
            context_turns=config["context_turns"],
            include_speaker=config["include_speaker"],
        )
        examples = normalize_examples(raw_examples)

    predictions, probabilities = predict_examples(
        examples,
        checkpoint,
        batch_size=16,
        max_length=128,
    )

    metrics = evaluate_predictions(examples, predictions)

    pred_df = save_experiment_outputs(
        name,
        examples,
        predictions,
        probabilities,
        metrics,
    )

    prediction_frames[name] = pred_df

    all_results.append({
        "experiment": name,
        "test_examples": len(examples),
        "accuracy": metrics["accuracy"],
        "macro_f1": metrics["macro_f1"],
        "weighted_f1": metrics["weighted_f1"],
    })

results_df = pd.DataFrame(all_results)
results_df


## 8. Error-analysis views

The saved `predictions.csv` files now contain the true label, predicted label, text/context, and class probabilities.

This lets us systematically inspect:

- false positives/false negatives by emotion,
- neutral absorption of minority emotions,
- examples corrected by speaker information,
- examples harmed by added context,
- high-confidence mistakes,
- low-confidence/ambiguous predictions.


In [ ]:
# Example: highest-confidence errors for the first available model
if prediction_frames:
    first_name = next(iter(prediction_frames))
    pred_df = prediction_frames[first_name]

    prob_cols = [f"prob_{label}" for label in LABELS]
    pred_df["prediction_confidence"] = pred_df[prob_cols].max(axis=1)

    high_conf_errors = (
        pred_df[~pred_df["correct"]]
        .sort_values("prediction_confidence", ascending=False)
        .head(20)
    )

    display(
        high_conf_errors[
            [c for c in [
                "dialogue_id",
                "utterance_id",
                "speaker",
                "text",
                "true_emotion",
                "predicted_emotion",
                "prediction_confidence",
            ] if c in high_conf_errors.columns]
        ]
    )


In [ ]:
# Cross-model comparison on the matched Context-1 target set.
# This is the appropriate comparison for Context-1 vs utterance-only.

if "transformer_context_1" in prediction_frames:
    ctx1 = prediction_frames["transformer_context_1"]

    # Context examples should retain target identifiers from the existing builder.
    id_pairs = [c for c in ["dialogue_id", "utterance_id"] if c in ctx1.columns]

    if len(id_pairs) == 2 and "transformer_baseline" in prediction_frames:
        base = prediction_frames["transformer_baseline"]

        merged = ctx1.merge(
            base[
                id_pairs + [
                    "true_emotion",
                    "predicted_emotion",
                    "correct",
                ]
            ].rename(columns={
                "predicted_emotion": "baseline_predicted_emotion",
                "correct": "baseline_correct",
            }),
            on=id_pairs,
            suffixes=("", "_baseline"),
        )

        merged["context_corrected"] = (
            (~merged["baseline_correct"]) & (merged["correct"])
        )
        merged["context_harmed"] = (
            (merged["baseline_correct"]) & (~merged["correct"])
        )

        print("Baseline wrong → Context-1 correct:",
              int(merged["context_corrected"].sum()))
        print("Baseline correct → Context-1 wrong:",
              int(merged["context_harmed"].sum()))

        display(
            merged[
                merged["context_corrected"] | merged["context_harmed"]
            ][
                id_pairs
                + ["text", "true_emotion",
                   "baseline_predicted_emotion",
                   "predicted_emotion",
                   "baseline_correct", "correct"]
            ].head(30)
        )


## 9. Save the consolidated Week 6 table

This produces a compact result artifact that can be committed to GitHub.


In [ ]:
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

if not results_df.empty:
    results_df.to_csv(
        TABLES_DIR / "week6_transformer_comparison.csv",
        index=False,
    )

results_df
